# Notebook 02 — Data Cleaning & EDA

**Retail Sales AI Analytics — IBM SkillsBuild Capstone**

This notebook covers:
- Applying the cleaning pipeline
- Feature engineering
- Exploratory Data Analysis: sales, profit, regions, sub-categories
- Discount-profit relationship

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)

from src.data_loader import load_raw_orders
from src.data_cleaning import clean_orders
from src.feature_engineering import engineer_features, save_processed
from src.analytics import (
    sales_summary, sales_by_year, monthly_sales_trend,
    sales_by_category, sales_by_subcategory, sales_by_region,
    profitability_by_subcategory, discount_profit_analysis
)

In [ ]:
# Load, clean, engineer
df_raw = load_raw_orders()
df_clean = clean_orders(df_raw)
df = engineer_features(df_clean)
save_processed(df)
print('Processed columns:', list(df.columns))
df.head(3)

In [ ]:
# KPI Summary
kpis = sales_summary(df)
for k, v in kpis.items():
    print(f'  {k:<35}: {v}')

In [ ]:
# Year-over-year performance
yoy = sales_by_year(df)
print(yoy.to_string(index=False))

In [ ]:
# Monthly sales trend
monthly = monthly_sales_trend(df)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['Year_Month'], monthly['Sales']/1e3, label='Sales ($k)', color='steelblue')
ax.plot(monthly['Year_Month'], monthly['Profit']/1e3, label='Profit ($k)', color='green', linestyle='--')
step = max(1, len(monthly)//12)
ax.set_xticks(range(0, len(monthly), step))
ax.set_xticklabels(monthly['Year_Month'].iloc[::step], rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Sales and Profit Trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sales and Profit by Category
cat = sales_by_category(df)
print(cat.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(cat['Category'], cat['Sales'], color='steelblue')
axes[0].set_title('Sales by Category')
axes[1].bar(cat['Category'], cat['Profit'], color='seagreen')
axes[1].set_title('Profit by Category')
plt.tight_layout()
plt.show()

In [ ]:
# Sub-Category Profitability
sub = profitability_by_subcategory(df).sort_values('Profit')
colors = ['red' if v < 0 else 'green' for v in sub['Profit']]
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(sub['Sub_Category'], sub['Profit']/1e3, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Profit (USD thousands)')
ax.set_title('Profit by Sub-Category (red = loss)')
plt.tight_layout()
plt.show()

In [ ]:
# Region Performance
reg = sales_by_region(df)
print(reg.to_string(index=False))

In [ ]:
# Discount vs Profit Margin
disc = discount_profit_analysis(df)
print(disc.to_string(index=False))

colors = ['red' if v < 0 else 'green' for v in disc['Avg_Profit_Margin']]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(disc['Discount_Bin'].astype(str), disc['Avg_Profit_Margin']*100, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Profit Margin by Discount Level')
ax.set_ylabel('Avg Profit Margin (%)')
plt.tight_layout()
plt.show()